# 📘Домашнє завдання №7. Основи баз даних та SQL

**Лекція 13. Основи баз даних та SQL**

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW7

## Завдання 1. Створення бази даних

Створіть локальну базу даних:

`store.db`

У базі створіть **4 таблиці**.

### Таблиця `P`

| поле | тип |
|---|---|
| id | INTEGER PRIMARY KEY |
| name | TEXT |
| city | TEXT |
| age | INTEGER |

### Таблиця `products`

| поле | тип |
|---|---|
| id | INTEGER PRIMARY KEY |
| product_name | TEXT |
| category | TEXT |
| price | REAL |

### Таблиця `orders`

| поле | тип |
|---|---|
| id | INTEGER PRIMARY KEY |
| customer_id | INTEGER |
| order_date | TEXT |

### Таблиця `order_items`

| поле | тип |
|---|---|
| id | INTEGER PRIMARY KEY |
| order_id | INTEGER |
| product_id | INTEGER |
| quantity | INTEGER |

### Зв'язки між таблицями

- customers → orders  
- orders → order_items  
- products → order_items

In [99]:
import sqlite3
import pandas as pd

# Input data
db_file_name = "store.db"

# Solution
!rm -f "$db_file_name"

# SQL
conn = sqlite3.connect(db_file_name)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers
(
    id   INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT    NOT NULL DEFAULT 'Undefined',
    city TEXT    NOT NULL DEFAULT 'Undefined',
    age  INTEGER NOT NULL DEFAULT 18 CHECK ( age >= 18 )
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS products
(
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL DEFAULT 'Undefined',
    category     TEXT NOT NULL DEFAULT 'Undefined',
    price        REAL NOT NULL DEFAULT 0.0 CHECK ( price >= 0.0 )
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders
(
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date  TEXT    NOT NULL DEFAULT (date('now')) CHECK (order_date IS date(order_date)),
    FOREIGN KEY (customer_id) REFERENCES customers (id) ON DELETE RESTRICT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items
(
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity   INTEGER NOT NULL DEFAULT 0 CHECK ( quantity >= 0 ),
    FOREIGN KEY (order_id) REFERENCES orders (id) ON DELETE RESTRICT,
    FOREIGN KEY (product_id) REFERENCES products (id) ON DELETE RESTRICT
);
""")

# Extract data from DB
cursor.execute("""
SELECT name AS tables
FROM sqlite_master
WHERE type='table' AND name NOT LIKE 'sqlite%';
""")
# Extract column names
columns_names = [column_name[0] for column_name in cursor.description]
# Extract data
table_data = cursor.fetchall()

conn.commit()
conn.close()

# Print results
df = (pd.DataFrame(table_data, columns=columns_names).
        style.set_caption("Table list").
        hide(axis='index'))
display(df)

tables
customers
products
orders
order_items


## Завдання 2. Наповнення таблиць даними

Заповніть таблиці тестовими даними.

### Таблиця `customers`
Додайте **мінімум 6 клієнтів**.

Приклад:

| id | name | city | age |
|---|---|---|---|
|1|Anna|Kyiv|21|
|2|Ivan|Lviv|25|

---

### Таблиця `products`
Додайте **мінімум 6 товарів**.

Приклад категорій:

- Electronics  
- Clothing  
- Books  

---

### Таблиця `orders`
Додайте **мінімум 8 замовлень**.

---

### Таблиця `order_items`
Додайте **10–12 записів**.

Зверніть увагу:

- одне замовлення може містити **кілька товарів**
- один товар може бути в **багатьох замовленнях**

In [100]:
import jinja2
import numpy as np
from faker import Faker
import faker_commerce
from transliterate import translit

# Input data
n_customers = 6
n_orders = 8
n_order_items = 12
n_max_items = 3
age_limit = (18, 100)
logic_map = {
    "Electronics": (["Smartphone", "Laptop", "Wireless Headphones"], (150.0, 1200.0)),
    "Appliances": (["Coffee Maker", "Toaster", "Blender"], (40.0, 250.0)),
    "Clothing": (["Cotton T-Shirt", "Leather Jacket", "Running Shoes"], (15.0, 150.0)),
    "Books": (["Science Fiction Novel", "Python Programming Guide", "World Atlas"], (10.0, 60.0)),
    "Home Decor": (["Ceramic Vase", "Scented Candle", "Wall Clock"], (5.0, 80.0)),
    "Fitness": (["Yoga Mat", "Dumbbell Set", "Jump Rope"], (12.0, 100.0))
}

# Solution
np.random.seed(42)
Faker.seed(42)
fake = Faker('uk_UA')
fake.add_provider(faker_commerce.Provider)

names = [translit(fake.first_name(), 'uk', reversed=True) for _ in range(n_customers)]
cities = [translit(fake.city_name(), 'uk', reversed=True) for _ in range(n_customers)]
ages = [int(np.random.randint(age_limit[0], age_limit[1] + 1)) for _ in range(n_customers)]

customers = list(zip(names, cities, ages))

# Logic in choosing
product_names = []
categories = []
prices = []
for category, (products, price_range) in logic_map.items():
    categories.append(category)
    product_names.append(str(np.random.choice(products)))
    prices.append(round(np.random.uniform(price_range[0], price_range[1]), 2))

# # No logic in choosing
# product_names = [fake.ecommerce_name() for _ in range(n_customers)]
# categories = [fake.ecommerce_category() for _ in range(n_customers)]
# prices = [round(fake.ecommerce_price() / 100000.0, 2)  for _ in range(n_customers)]

products = list(zip(product_names, categories, prices))

customer_ids = []
order_dates = []
for _ in range(n_orders):
    customer_ids.append(int(np.random.randint(n_customers) + 1))
    order_dates.append(fake.date_between(start_date='-1y', end_date='today').strftime('%Y-%m-%d'))

orders = list(zip(customer_ids, order_dates))

order_ids = []
product_ids = []
quantities = []
for _ in range(n_order_items):
    order_ids.append(int(np.random.randint(n_customers) + 1))
    product_ids.append(int(np.random.randint(len(products)) + 1))
    quantities.append(int(np.random.randint(n_max_items) + 1))

order_items = list(zip(order_ids, product_ids, quantities))

# SQL
conn = sqlite3.connect(db_file_name)
cursor = conn.cursor()

cursor.executemany("""
INSERT INTO customers (name, city, age)
VALUES (?, ?, ?)
""", customers)

cursor.executemany("""
INSERT INTO products (product_name, category, price)
VALUES (?, ?, ?)
""", products)

cursor.executemany("""
INSERT INTO orders (customer_id, order_date)
VALUES (?, ?)
""", orders)

cursor.executemany("""
INSERT INTO order_items (order_id, product_id, quantity)
VALUES (?, ?, ?)
""", order_items)

# Extract data from DB
cursor.execute("""
SELECT *
FROM customers;
""")
# Extract column names
columns_names_1 = [column_name[0] for column_name in cursor.description]
# Extract data
table_data_1 = cursor.fetchall()

# Extract data from DB
cursor.execute("""
SELECT *
FROM products;
""")
# Extract column names
columns_names_2 = [column_name[0] for column_name in cursor.description]
# Extract data
table_data_2 = cursor.fetchall()

# Extract data from DB
cursor.execute("""
SELECT *
FROM orders;
""")
# Extract column names
columns_names_3 = [column_name[0] for column_name in cursor.description]
# Extract data
table_data_3 = cursor.fetchall()

# Extract data from DB
cursor.execute("""
SELECT *
FROM order_items;
""")
# Extract column names
columns_names_4 = [column_name[0] for column_name in cursor.description]
# Extract data
table_data_4 = cursor.fetchall()

conn.commit()
conn.close()

# Print results
df_1 = (pd.DataFrame(table_data_1, columns=columns_names_1).
        style.set_caption("Table: customers").
        hide(axis='index'))
df_2 = (pd.DataFrame(table_data_2, columns=columns_names_2).
        style.set_caption("Table: products").
        hide(axis='index'))
df_3 = (pd.DataFrame(table_data_3, columns=columns_names_3).
        style.set_caption("Table: orders").
        hide(axis='index'))
df_4 = (pd.DataFrame(table_data_4, columns=columns_names_4).
        style.set_caption("Table: orders").
        hide(axis='index'))
display(df_1, df_2, df_3, df_4)


id,name,city,age
1,Milena,Verkhn'odniprovs'k,69
2,Vitalij,Sosnivka,32
3,Al'bert,Borshchiv,89
4,Martyn,Sambir,78
5,Leonid,Staryj Sambir,38
6,Klyment,Jahotyn,100


id,product_name,category,price
1,Wireless Headphones,Electronics,210.990000
2,Coffee Maker,Appliances,166.230000
3,Running Shoes,Clothing,17.780000
4,Python Programming Guide,Books,46.100000
5,Scented Candle,Home Decor,20.930000
6,Yoga Mat,Fitness,66.340000


id,customer_id,order_date
1,2,2025-10-10
2,6,2025-10-27
3,5,2025-04-05
4,4,2025-04-28
5,1,2025-06-18
6,1,2025-10-31
7,3,2025-10-16
8,3,2025-12-12


id,order_id,product_id,quantity
1,2,4,3
2,6,6,3
3,6,3,3
4,4,1,3
5,5,3,3
6,5,1,3
7,2,4,1
8,4,6,2
9,2,1,2
10,5,2,3


## Завдання 3. JOIN кількох таблиць

Отримайте список усіх покупок у магазині.

Вивести потрібно:

customer_name | product_name | category | quantity | price


Для цього потрібно об'єднати **4 таблиці**:

- customers  
- orders  
- order_items  
- products  

Використати:

- `SELECT`
- `JOIN`

## Завдання 4. Фільтрація покупок

Показати всі покупки товарів категорії **Electronics**.

Вивести:

customer_name | product_name | price

Використати:

- JOIN
- WHERE

## Завдання 5. Описова статистика по товарах

Порахуйте **скільки разів кожен товар був куплений**.

Вивести:

product_name | total_sales

Використати:

- SUM
- GROUP BY
- JOIN

## Завдання 6. Унікальні покупці по категоріях

Порахуйте **кількість унікальних покупців у кожній категорії товарів**.

Вивести:

category | unique_customers

Використати:

- COUNT(DISTINCT)
- JOIN
- GROUP BY

## Завдання 7. Популярність товарів (CASE)

Створіть колонку **sales_level**, яка визначає популярність товару.

Правила:

- $\geq$ 3 продажі → "High demand"
- 2 продажі → "Medium demand"
- 1 продаж → "Low demand"

Вивести:

product_name | sales_count | sales_level

Використати:

- CASE
- SUM
- GROUP BY
- JOIN

## Завдання 8. HAVING

Знайдіть **товари, які були куплені більше ніж 2 рази**.

Вивести:

product_name | sales_count

Використати:

- JOIN
- GROUP BY
- HAVING




## Завдання 9. Загальна виручка по категоріях

Порахуйте **загальну виручку для кожної категорії товарів**.

Формула виручки:

revenue = price * quantity

Вивести:

category | total_revenue

Використати:

- JOIN
- SUM
- GROUP BY

## Завдання 10. Активність клієнтів

Визначте **активність клієнтів за кількістю замовлень**.

Створіть нову колонку **customer_activity**:

- $\geq$ 3 замовлення → "Active"
- 2 замовлення → "Regular"
- 1 замовлення → "New"

Вивести:

customer_name | number_of_orders | customer_activity

Використати:

- JOIN
- COUNT
- GROUP BY
- CASE